# Phase B — HBO_revised: Width-First + J_topo Screening
## Strategy: Width-First Filter + J_topo Within Groups

**Round 1 failure:** HBO selected HIGH J_topo globally → picked narrow-deep (24/6) → capacity-limited.

**HBO_revised design:**
1. Width-first filter: only keep width ≥ 48 (removes capacity-limited narrow nets)
2. Within wide pool: select top-30 by J_topo HIGH
3. Same L1 (10ep) → top-5 → L2 (50ep) pipeline

**Hypothesis:** HBO_revised beats Random because:
- Width-first removes narrow-deep configs (capacity floor)
- J_topo HIGH within wide pool → better optimization → lower E_floor

**Width threshold W_min = 48**

In [ ]:
# TPU + Clone
import os, torch, subprocess
import torch_xla.core.xla_model as xm
import torch_xla.distributed.parallel_loader as pl
DEVICE = xm.xla_device()
_ = torch.zeros((2,3), device=DEVICE); print("TPU:", DEVICE)
os.chdir("/kaggle/working")
subprocess.run(["rm", "-rf", "ThermoRG-NN"], check=True)
from kaggle_secrets import UserSecretsClient
gh = UserSecretsClient().get_secret("gh_token")
subprocess.run(["git", "clone", f"https://{gh}@github.com/xliu203/ThermoRG-NN.git", "--branch", "develop"], check=True)
subprocess.run(["git", "config", "--global", "user.email", "xliu203@asu.edu"], check=True)
subprocess.run(["git", "config", "--global", "user.name", "Leo"], check=True)
import sys; sys.path.insert(0, "/kaggle/working/ThermoRG-NN")
print("Cloned.")

In [ ]:
# Imports
import math, time, json, random, warnings, os, numpy as np
from pathlib import Path
import torch, torch.nn as nn
from scipy.stats import spearmanr
import torchvision.transforms as T
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader
from thermorg import compute_J_topo
print("Imports OK.")

In [ ]:
# ThermoNet
class ConvBlock(nn.Module):
    def __init__(self, ci, co, norm='none', skip=False, stride=1):
        super().__init__()
        self.conv = nn.Conv2d(ci, co, 3, padding=1, stride=stride, bias=False)
        if norm == 'batchnorm': self.norm = nn.BatchNorm2d(co)
        elif norm == 'layernorm': self.norm = nn.LayerNorm([co, 32//stride, 32//stride])
        else: self.norm = nn.Identity()
        self.act = nn.GELU()
        self.skip = skip and stride == 1 and ci == co
    def forward(self, x):
        out = self.act(self.norm(self.conv(x)))
        if self.skip: out = out + x
        return out

class ThermoNet(nn.Module):
    def __init__(self, base_ch=64, depth=3, norm='none', skip=False):
        super().__init__()
        ch = [3] + [base_ch]*depth
        self.blocks = nn.ModuleList([ConvBlock(ch[i], ch[i+1], norm, skip and i>0) for i in range(depth)])
        self.pool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(ch[-1], 10)
    def forward(self, x):
        for b in self.blocks: x = b(x)
        return self.fc(self.pool(x).squeeze())

def get_J_topo(base_ch, depth, norm, skip):
    model = ThermoNet(base_ch=base_ch, depth=depth, norm=norm, skip=skip)
    J, _ = compute_J_topo(model)
    del model
    return J
print("ThermoNet OK.")

In [ ]:
# DataLoaders
BATCH=256; LR=0.01; WD=5e-4; MOM=0.9
OUT=Path("/kaggle/working/phase_b_hbo_revised"); OUT.mkdir(exist_ok=True)
CKPT=OUT/"ckpt"; CKPT.mkdir(exist_ok=True)
tfm = T.Compose([T.RandomCrop(32,padding=4),T.RandomHorizontalFlip(),T.ToTensor(),T.Normalize([.4914,.4822,.4465],[.2470,.2435,.2616])])
tvf = T.Compose([T.ToTensor(),T.Normalize([.4914,.4822,.4465],[.2470,.2435,.2616])])
train_ds = CIFAR10("./data", True, tfm, download=True)
val_ds   = CIFAR10("./data", False, tvf, download=True)
def loaders(b=BATCH):
    return (
        DataLoader(train_ds, batch_size=b, shuffle=True, num_workers=4, drop_last=True, pin_memory=True),
        DataLoader(val_ds,   batch_size=b, shuffle=False, num_workers=4, drop_last=True, pin_memory=True)
    )
print(f"Data: {len(train_ds)} train, {len(val_ds)} val")

In [ ]:
# Training function
def train(model, tr, va, ep, lr=LR, wd=WD, mom=MOM):
    model = model.to(DEVICE)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=mom, weight_decay=wd)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=ep)
    crit = nn.CrossEntropyLoss()
    best_l, best_a, best_e = float('inf'), 0.0, 0
    for e in range(ep):
        model.train()
        for X, y in pl.MpDeviceLoader(tr, DEVICE):
            opt.zero_grad()
            loss = crit(model(X), y)
            loss.backward()
            xm.optimizer_step(opt, barrier=True)
        sch.step()
        model.eval()
        ls_t=torch.tensor(0.,device=DEVICE); ca_t=torch.tensor(0.,device=DEVICE); to_t=torch.tensor(0.,device=DEVICE)
        with torch.no_grad():
            for X, y in pl.MpDeviceLoader(va, DEVICE):
                o = model(X)
                ls_t += crit(o,y)*X.size(0); ca_t += (o.argmax(1)==y).float().sum(); to_t += X.size(0)
        vl = (ls_t/to_t).item(); acc = (ca_t/to_t).item()
        if vl < best_l: best_l, best_a, best_e = vl, acc, e+1
    del model
    return {"best_val_loss": best_l, "best_val_acc": best_a, "best_epoch": best_e}
print("train() OK.")

---
## Candidates + J_topo

In [ ]:
# Stratified candidates (same as Round 1)
WIDTHS=[16,24,32,48,64,96]; DEPTHS=[3,4,5,6]; NORMS=['none','batchnorm','layernorm']; SKIPS=[False,True]
all_cfgs=[{"width":w,"depth":d,"norm":n,"skip":s} for w in WIDTHS for d in DEPTHS for n in NORMS for s in SKIPS]
random.seed(42)
stratified=[]
for n in NORMS:
    for s in SKIPS:
        pool=[c for c in all_cfgs if c["norm"]==n and c["skip"]==s]
        stratified.extend(random.sample(pool, min(8,len(pool))))
remaining=[c for c in all_cfgs if c not in stratified]
stratified.extend(random.sample(remaining, 100-len(stratified)))
CANDIDATES=stratified
print(f"Candidates: {len(CANDIDATES)}, none={sum(1 for c in CANDIDATES if c['norm']=='none')}, bn={sum(1 for c in CANDIDATES if c['norm']=='batchnorm')}, ln={sum(1 for c in CANDIDATES if c['norm']=='layernorm')}")

In [ ]:
# J_topo
print("Computing J_topo..."); t0=time.time()
for c in CANDIDATES: c["J_topo"] = get_J_topo(c["width"], c["depth"], c["norm"], c["skip"])
print(f"Done in {time.time()-t0:.1f}s, J_topo range=[{min(c['J_topo'] for c in CANDIDATES):.4f}, {max(c['J_topo'] for c in CANDIDATES):.4f}]")

---
## HBO_revised: Width-First Filter (W >= 48) + J_topo HIGH

In [ ]:
# Width-first filter: W >= 48
W_MIN=48
wide_pool=[c for c in CANDIDATES if c["width"]>=W_MIN]
narrow=[c for c in CANDIDATES if c["width"]<W_MIN]
print(f"Wide pool (W>={W_MIN}): {len(wide_pool)} configs")
print(f"Narrow pool removed: {len(narrow)} configs")

# HBO_revised: top-30 by J_topo HIGH within wide pool
HBO30=sorted(wide_pool, key=lambda x: x["J_topo"], reverse=True)[:30]
print(f"\nHBO_revised top-30: J_topo [{HBO30[-1]['J_topo']:.4f}, {HBO30[0]['J_topo']:.4f}]")
print(f"  widths: {sorted(set(c['width'] for c in HBO30))}")
print(f"  norms:  {sorted(set(c['norm'] for c in HBO30))}")

# Random: 30 random from full pool
RAND30=random.sample(CANDIDATES, 30)
print(f"\nRandom 30: widths {sorted(set(c['width'] for c in RAND30))}")
print(f"Random J_topo mean={np.mean([c['J_topo'] for c in RAND30]):.4f} vs HBO_revised={np.mean([c['J_topo'] for c in HBO30]):.4f}")

---
## Stage 1: L1 (10 epochs)

In [ ]:
# Random L1
print("="*60); print("Random L1 (30 x 10ep)"); print("="*60)
RAND_RESULTS=[]
t0=time.time()
for i,c in enumerate(RAND30):
    orig=CANDIDATES.index(c); jf=OUT/f"rand_l1_{orig}.json"
    if jf.exists():
        with open(jf) as f: r=json.load(f); RAND_RESULTS.append(r)
        print(f"  [{i+1}/30] idx={orig} [SKIP] loss={r['best_val_loss']:.4f}")
        continue
    print(f"  [{i+1}/30] idx={orig} {c['width']}/{c['depth']}/{c['norm']}/{c['skip']} J={c['J_topo']:.4f}...", end="", flush=True)
    torch.manual_seed(42+i); model=ThermoNet(base_ch=c['width'],depth=c['depth'],norm=c['norm'],skip=c['skip'])
    tr,va=loaders(); r=train(model,tr,va,10)
    r["config"]=c; RAND_RESULTS.append(r)
    with open(jf,"w") as f: json.dump(r,f)
    print(f" loss={r['best_val_loss']:.4f}")
print(f"\nDone in {(time.time()-t0)/60:.1f} min")

In [ ]:
# HBO_revised L1
print("="*60); print("HBO_revised L1 (30 x 10ep)"); print("="*60)
HBO_RESULTS=[]
t0=time.time()
for i,c in enumerate(HBO30):
    orig=CANDIDATES.index(c); jf=OUT/f"hbo_l1_{orig}.json"
    if jf.exists():
        with open(jf) as f: r=json.load(f); HBO_RESULTS.append(r)
        print(f"  [{i+1}/30] idx={orig} [SKIP] loss={r['best_val_loss']:.4f}")
        continue
    print(f"  [{i+1}/30] idx={orig} {c['width']}/{c['depth']}/{c['norm']}/{c['skip']} J={c['J_topo']:.4f}...", end="", flush=True)
    torch.manual_seed(42+orig+1000); model=ThermoNet(base_ch=c['width'],depth=c['depth'],norm=c['norm'],skip=c['skip'])
    tr,va=loaders(); r=train(model,tr,va,10)
    r["config"]=c; HBO_RESULTS.append(r)
    with open(jf,"w") as f: json.dump(r,f)
    print(f" loss={r['best_val_loss']:.4f}")
print(f"\nDone in {(time.time()-t0)/60:.1f} min")

---
## Stage 2: Top-5 + L2 (50 epochs)

In [ ]:
# Top-5 by L1
RAND_TOP5=sorted(RAND_RESULTS,key=lambda x:x['best_val_loss'])[:5]
HBO_TOP5=sorted(HBO_RESULTS,key=lambda x:x['best_val_loss'])[:5]
print("Random top-5:")
for r in RAND_TOP5: c=r['config']; print(f"  L1={r['best_val_loss']:.4f} J={c['J_topo']:.4f} {c['width']}/{c['depth']}/{c['norm']}/{c['skip']}")
print("\nHBO_revised top-5:")
for r in HBO_TOP5: c=r['config']; print(f"  L1={r['best_val_loss']:.4f} J={c['J_topo']:.4f} {c['width']}/{c['depth']}/{c['norm']}/{c['skip']}")

In [ ]:
# Random L2
print("="*60); print("Random L2 top-5 (50ep)"); print("="*60)
RAND_L2=[]
for i,r in enumerate(RAND_TOP5):
    cfg=r['config']; orig=CANDIDATES.index(cfg); jf=OUT/f"rand_l2_{orig}.json"
    if jf.exists():
        with open(jf) as f: r2=json.load(f); RAND_L2.append(r2)
        print(f"  [{i+1}/5] idx={orig} [SKIP] loss={r2['best_val_loss']:.4f}")
        continue
    print(f"  [{i+1}/5] idx={orig}..."); torch.manual_seed(42+orig)
    model=ThermoNet(base_ch=cfg['width'],depth=cfg['depth'],norm=cfg['norm'],skip=cfg['skip'])
    tr,va=loaders(); r2=train(model,tr,va,50)
    r2["config"]=cfg; RAND_L2.append(r2)
    with open(jf,"w") as f: json.dump(r2,f)
    print(f"  loss={r2['best_val_loss']:.4f} acc={r2['best_val_acc']:.4f}")

In [ ]:
# HBO_revised L2
print("="*60); print("HBO_revised L2 top-5 (50ep)"); print("="*60)
HBO_L2=[]
for i,r in enumerate(HBO_TOP5):
    cfg=r['config']; orig=CANDIDATES.index(cfg); jf=OUT/f"hbo_l2_{orig}.json"
    if jf.exists():
        with open(jf) as f: r2=json.load(f); HBO_L2.append(r2)
        print(f"  [{i+1}/5] idx={orig} [SKIP] loss={r2['best_val_loss']:.4f}")
        continue
    print(f"  [{i+1}/5] idx={orig}..."); torch.manual_seed(42+orig+1000)
    model=ThermoNet(base_ch=cfg['width'],depth=cfg['depth'],norm=cfg['norm'],skip=cfg['skip'])
    tr,va=loaders(); r2=train(model,tr,va,50)
    r2["config"]=cfg; HBO_L2.append(r2)
    with open(jf,"w") as f: json.dump(r2,f)
    print(f"  loss={r2['best_val_loss']:.4f} acc={r2['best_val_acc']:.4f}")

---
## Final Results

In [ ]:
# Final comparison
rl2=[r['best_val_loss'] for r in RAND_L2]; hl2=[r['best_val_loss'] for r in HBO_L2]
print("="*60)
print("FINAL: HBO_revised vs Random (50-epoch)")
print("="*60)
print(f"Random best={min(rl2):.4f}, mean={np.mean(rl2):.4f}")
print(f"HBO_revised best={min(hl2):.4f}, mean={np.mean(hl2):.4f}")
delta=min(rl2)-min(hl2)
print(f"Delta (Random-HBO_revised): {delta:.4f}")
if delta>0.01: print(">>> HBO_revised WINS <<<")
elif delta<-0.01: print(">>> Random WINS <<<")
else: print(">>> TIE <<<")
print("\nRandom L2 top-5:")
for r in sorted(RAND_L2,key=lambda x:x['best_val_loss']): c=r['config']
    print(f"  {r['best_val_loss']:.4f} J={c['J_topo']:.4f} {c['width']}/{c['depth']}/{c['norm']}/{c['skip']}")
print("\nHBO_revised L2 top-5:")
for r in sorted(HBO_L2,key=lambda x:x['best_val_loss']): c=r['config']
    print(f"  {r['best_val_loss']:.4f} J={c['J_topo']:.4f} {c['width']}/{c['depth']}/{c['norm']}/{c['skip']}")

In [ ]:
# Save results + push
final={"strategy":"HBO_revised: width-first(W>=48) + J_topo HIGH","W_min":W_MIN,
       "random_l2":[{"loss":r["best_val_loss"],"config":r["config"]} for r in RAND_L2],
       "hbo_l2":[{"loss":r["best_val_loss"],"config":r["config"]} for r in HBO_L2],
       "random_best":float(min(rl2)),"random_mean":float(np.mean(rl2)),
       "hbo_best":float(min(hl2)),"hbo_mean":float(np.mean(hl2)),"delta":float(delta)}
with open(OUT/"results.json","w") as f: json.dump(final,f,indent=2)
print("Results saved.")

In [ ]:
# Git push
os.chdir("/kaggle/working/ThermoRG-NN")
subprocess.run(["git","add","experiments/phase_b/notebooks/phase_b_hbo_revised.ipynb"],check=True)
subprocess.run(["git","commit","-m","Add HBO_revised: width-first + J_topo screening (Phase B Round 2)"],check=True)
result=subprocess.run(["git","push","origin","develop"],capture_output=True,text=True)
print(result.stdout[-500:] if result.stdout else "")
if result.returncode!=0: print(result.stderr[-500:])